# s04 line by line: from GRAPH files to a trained model

This notebook follows `scripts/python/s04_train_model.py` **function by function and line by line**, on tiny toy data.
Nothing here touches the real data on scratch.

**How every section is built**
1. **The function** — a copy of the s04 function (same name, same logic, docstrings left out). Running the cell only
   defines it.
2. **Line by line** — the body of the function taken apart: every cell runs one line (or a few lines that belong
   together) on real toy inputs and prints what came out (type, dtype, shape, values). The comment at the top of
   each cell says what the line does and why.
3. **Check** — the step-by-step result is compared with a call of the function itself, so you know the breakdown is
   exactly what the function does.

| Section | s04 function(s) |
|---|---|
| 1 | (s03 output) the toy GRAPH files |
| 2 | `build_pool_index`, `sample_pools` |
| 3 | `load_pools`, `load_norm_stats` |
| 4 | `draw_subset`, `pairwise_geodesic`, `knn_edges`, `dequantise_gse`, `build_node_features`, `build_edge_features`, target |
| 5 | `PoolDataset`, `estimate_edge_stats` |
| 6 | `DataLoader` (batching) |
| 7 | `InSARGNN` |
| 8 | `loss_fn` |
| 9 | `set_seed`, `train_one_epoch`, `evaluate`, `save_checkpoint`, `train_model`, `evaluate_on_dev` |

Kernel: **LOSTInPLACE Tutorial** (this folder's uv environment, see `README.md`). Run the cells from top to bottom:
later sections use the objects of earlier ones.

## 0. Setup

In [1]:
import json
import random
import tempfile
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from pyproj import Geod
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import to_undirected

# geodesic distances and bearings on the WGS84 ellipsoid, as in s04
GEOD = Geod(ellps="WGS84")

torch.manual_seed(0)
np.set_printoptions(precision=4, suppress=True, linewidth=140)   # suppress: no 1e-05 notation
pl.Config.set_tbl_cols(12)

def show(name, obj, **printoptions):
    # prints what an object is (type, dtype, shape) and then its value; printoptions e.g. precision=6 for coordinates
    info = type(obj).__name__
    if hasattr(obj, "dtype"):
        info += f" | dtype {obj.dtype}"
    if hasattr(obj, "shape"):
        info += f" | shape {tuple(obj.shape)}"
    print(f"{name}: {info}")
    with np.printoptions(**printoptions):
        print(obj, "\n")

print("torch", torch.__version__, "| polars", pl.__version__, "| cuda available:", torch.cuda.is_available())

/dss/dsshome1/0C/di54haf/LOSTInPLACE/scripts/python/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch 2.14.0+cu130 | polars 1.44.2 | cuda available: False


## 1. Toy GRAPH files (what s03 writes)

s03 writes one `<burst>_GRAPH.parquet` per burst into `data/<set>/<location>/<orbit>/`. Every row is one EGMS point that
belongs to a pool. A pool = all rows with the same `seed_pid`; exactly one row of it has `seed == 1` (the seed).

The cell below builds the same structure in a temporary folder: 2 train regions and 1 dev region, a few pools each,
with a smooth fake velocity field and random fake GSE vectors. This is toy-data code, not s04 — you don't have to go
through it line by line.

In [2]:
ROOT = Path(tempfile.mkdtemp()) / "data"
rng = np.random.default_rng(0)

def toy_velocity(lat, lon):
    # a smooth fake deformation field in mm/yr: a subsidence bowl plus a gentle trend
    return -15 * np.exp(-((lat - 48.10) ** 2 + (lon - 11.50) ** 2) / 0.0004) + 20 * (lon - 11.5)

def quantise_gse(x):
    # the inverse of the de-quantisation in s04: unit vectors -> int8, like the real GSE columns
    return np.clip(np.round(np.sign(x) * np.sqrt(np.abs(x)) * 127.5), -127, 127).astype(np.int8)

def make_graph_file(path, n_pools, centre, prefix):
    rows = []
    for p in range(n_pools):
        n = int(rng.integers(5, 27))                      # pool size incl. the seed (some below min_graph_size = 7)
        seed_lat = centre[0] + rng.normal(0, 0.01); seed_lon = centre[1] + rng.normal(0, 0.01)
        lat = seed_lat + np.r_[0, rng.normal(0, 0.005, n - 1)]   # first point = the seed, the others within ~1 km
        lon = seed_lon + np.r_[0, rng.normal(0, 0.007, n - 1)]
        gse = rng.normal(size=(n, 64)); gse /= np.linalg.norm(gse, axis=1, keepdims=True)
        seed_pid = f"{prefix}_{p}_0"
        for j in range(n):
            row = {"pid": f"{prefix}_{p}_{j}", "seed_pid": seed_pid, "seed": np.int8(j == 0),
                   "latitude": lat[j], "longitude": lon[j], "easting": 0.0, "northing": 0.0,
                   "incidence_angle": 38 + rng.normal(0, 3),
                   "mean_velocity": toy_velocity(lat[j], lon[j]) + rng.normal(0, 1)}
            row.update({f"gse_{i}": v for i, v in enumerate(quantise_gse(gse[j]))})
            rows.append(row)
    df = pl.DataFrame(rows).with_columns(pl.col("seed").cast(pl.Int8), pl.col([f"gse_{i}" for i in range(64)]).cast(pl.Int8))
    path.parent.mkdir(parents=True, exist_ok=True)
    df.write_parquet(path)
    return str(path)

train_files = [make_graph_file(ROOT / "train" / "RegionA" / "ascending" / "burstA1_GRAPH.parquet", 12, (48.10, 11.50), "A1"),
               make_graph_file(ROOT / "train" / "RegionA" / "descending" / "burstA2_GRAPH.parquet", 10, (48.10, 11.50), "A2"),
               make_graph_file(ROOT / "train" / "RegionB" / "ascending" / "burstB1_GRAPH.parquet", 12, (48.12, 11.47), "B1")]
dev_files = [make_graph_file(ROOT / "dev" / "RegionC" / "ascending" / "burstC1_GRAPH.parquet", 8, (48.11, 11.49), "C1")]
print("train files:", *train_files, sep="\n  ")
print("dev files:", *dev_files, sep="\n  ")

train files:
  /dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionA/ascending/burstA1_GRAPH.parquet
  /dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionA/descending/burstA2_GRAPH.parquet
  /dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionB/ascending/burstB1_GRAPH.parquet
dev files:
  /dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/dev/RegionC/ascending/burstC1_GRAPH.parquet


In [3]:
# one GRAPH file: columns, dtypes, number of rows and pools
g = pl.read_parquet(train_files[0])
print("shape (rows, columns):", g.shape)
print("schema (first 9 columns):", dict(list(g.schema.items())[:9]))
print("GSE column dtype:", g["gse_0"].dtype)
print("pools in this file:", g["seed_pid"].n_unique(), "| seeds:", int(g["seed"].sum()))
g.select("pid", "seed_pid", "seed", "latitude", "longitude", "incidence_angle", "mean_velocity", "gse_0", "gse_1").head(6)

shape (rows, columns): (180, 73)
schema (first 9 columns): {'pid': String, 'seed_pid': String, 'seed': Int8, 'latitude': Float64, 'longitude': Float64, 'easting': Float64, 'northing': Float64, 'incidence_angle': Float64, 'mean_velocity': Float64}
GSE column dtype: Int8
pools in this file: 12 | seeds: 12


pid,seed_pid,seed,latitude,longitude,incidence_angle,mean_velocity,gse_0,gse_1
str,str,i8,f64,f64,f64,f64,i8,i8
"""A1_0_0""","""A1_0_0""",1,48.098679,11.506404,35.480716,-13.377433,62,59
"""A1_0_1""","""A1_0_0""",0,48.099203,11.507062,37.819603,-13.481498,15,-13
"""A1_0_2""","""A1_0_0""",0,48.096001,11.5012,33.826558,-16.043427,-17,-51
"""A1_0_3""","""A1_0_0""",0,48.100487,11.499952,38.383953,-14.813778,-47,50
"""A1_0_4""","""A1_0_0""",0,48.105199,11.5032,44.549386,-13.78017,-88,23
"""A1_0_5""","""A1_0_0""",0,48.103414,11.507946,41.064272,-11.087843,-36,-39


In [4]:
# points per pool: the pool sizes vary (sparse areas give smaller pools)
g.group_by("seed_pid").len().sort("seed_pid")

seed_pid,len
str,u32
"""A1_0_0""",23
"""A1_10_0""",11
"""A1_11_0""",10
"""A1_1_0""",19
"""A1_2_0""",14
…,…
"""A1_5_0""",7
"""A1_6_0""",14
"""A1_7_0""",24


## 2. The pool index and the sample

### 2.1 `build_pool_index` — list the pools without loading their points

Instead of loading all points, s04 first only lists the pools: one row per pool, (`seed_pid`, `graph_file`). Every pool
has exactly one seed row, so the seed rows of a file are exactly its pools.

In [5]:
def build_pool_index(graph_files):
    tables = [pl.scan_parquet(path).filter(pl.col("seed") == 1).select("seed_pid")
              .with_columns(pl.lit(str(path)).alias("graph_file")).collect() for path in graph_files]
    return pl.concat(tables)

**Line by line.** The whole function is one list comprehension with a polars chain inside. We take the chain apart for one file, then do the loop and the concat.

In [6]:
# pl.scan_parquet(path): does NOT read the file. It returns a LazyFrame = a plan of what to read later.
# Nothing is in memory until .collect() runs at the end of the chain, so polars can optimise the whole chain at once.
path = train_files[0]
lazy = pl.scan_parquet(path)
show("lazy", lazy)
print("the file has", len(lazy.collect_schema().names()), "columns, the first ones:", lazy.collect_schema().names()[:8])

lazy: LazyFrame
naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Parquet SCAN [/dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionA/ascending/burstA1_GRAPH.parquet]
PROJECT */73 COLUMNS
ESTIMATED ROWS: 180 

the file has 73 columns, the first ones: ['pid', 'seed_pid', 'seed', 'latitude', 'longitude', 'easting', 'northing', 'incidence_angle']


In [7]:
# .filter(pl.col("seed") == 1): keep only the rows where the seed flag is 1 -> one row per pool.
# pl.col("seed") == 1 is an expression (a description of a computation), not a result yet.
expr = pl.col("seed") == 1
show("expr", expr)
lazy_seeds = lazy.filter(expr)
print("plan (read bottom-up):\n", lazy_seeds.explain())
print("\nwhat the filter keeps (collected only for the print):")
print(lazy_seeds.select("pid", "seed_pid", "seed").collect().head())

expr: Expr
[(col("seed")) == (dyn int: 1)] 

plan (read bottom-up):
 Parquet SCAN [/dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionA/ascending/burstA1_GRAPH.parquet]
PROJECT */73 COLUMNS
SELECTION: col("seed") == 1
ESTIMATED ROWS: 180

what the filter keeps (collected only for the print):
shape: (5, 3)
┌────────┬──────────┬──────┐
│ pid    ┆ seed_pid ┆ seed │
│ ---    ┆ ---      ┆ ---  │
│ str    ┆ str      ┆ i8   │
╞════════╪══════════╪══════╡
│ A1_0_0 ┆ A1_0_0   ┆ 1    │
│ A1_1_0 ┆ A1_1_0   ┆ 1    │
│ A1_2_0 ┆ A1_2_0   ┆ 1    │
│ A1_3_0 ┆ A1_3_0   ┆ 1    │
│ A1_4_0 ┆ A1_4_0   ┆ 1    │
└────────┴──────────┴──────┘


In [8]:
# .select("seed_pid"): of the seed rows only the pool id is needed
lazy_ids = lazy_seeds.select("seed_pid")
print(lazy_ids.collect().head())

shape: (5, 1)
┌──────────┐
│ seed_pid │
│ ---      │
│ str      │
╞══════════╡
│ A1_0_0   │
│ A1_1_0   │
│ A1_2_0   │
│ A1_3_0   │
│ A1_4_0   │
└──────────┘


In [9]:
# .with_columns(pl.lit(str(path)).alias("graph_file")): adds a column holding the same value in every row.
# pl.lit = a literal (constant) value, .alias = the column name. So every pool knows which file it comes from,
# which load_pools needs later to know where to read the pool's points.
lazy_index = lazy_ids.with_columns(pl.lit(str(path)).alias("graph_file"))
print(lazy_index.collect().head())

shape: (5, 2)
┌──────────┬─────────────────────────────────┐
│ seed_pid ┆ graph_file                      │
│ ---      ┆ ---                             │
│ str      ┆ str                             │
╞══════════╪═════════════════════════════════╡
│ A1_0_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_1_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_2_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_3_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_4_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
└──────────┴─────────────────────────────────┘


In [10]:
# .collect(): now the plan is executed and the file is actually read -> a (small) DataFrame
table = lazy_index.collect()
show("table", table)

table: DataFrame | shape (12, 2)
shape: (12, 2)
┌──────────┬─────────────────────────────────┐
│ seed_pid ┆ graph_file                      │
│ ---      ┆ ---                             │
│ str      ┆ str                             │
╞══════════╪═════════════════════════════════╡
│ A1_0_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_1_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_2_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_3_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_4_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ …        ┆ …                               │
│ A1_7_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_8_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_9_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_10_0  ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_11_0  ┆ /dss/dsstbyfs02/scratch/0C/di5… │
└──────────┴─────────────────────────────────┘ 



In [11]:
# the list comprehension does the same for every file of the list -> one small DataFrame per file
tables = [pl.scan_parquet(path).filter(pl.col("seed") == 1).select("seed_pid")
          .with_columns(pl.lit(str(path)).alias("graph_file")).collect() for path in train_files]
print("type:", type(tables).__name__, "| one table per file:", len(tables), "| rows per table:", [t.height for t in tables])

type: list | one table per file: 3 | rows per table: [12, 10, 12]


In [12]:
# pl.concat(tables): stacks the tables on top of each other -> one index over all train files
pool_index = pl.concat(tables)
show("pool_index", pool_index)
print("check, same as the function:", pool_index.equals(build_pool_index(train_files)))
train_index_all = pool_index

pool_index: DataFrame | shape (34, 2)
shape: (34, 2)
┌──────────┬─────────────────────────────────┐
│ seed_pid ┆ graph_file                      │
│ ---      ┆ ---                             │
│ str      ┆ str                             │
╞══════════╪═════════════════════════════════╡
│ A1_0_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_1_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_2_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_3_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_4_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ …        ┆ …                               │
│ B1_7_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_8_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_9_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_10_0  ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_11_0  ┆ /dss/dsstbyfs02/scratch/0C/di5… │
└──────────┴─────────────────────────────────┘ 

check, same as the function: True


### 2.2 `sample_pools` — keep `--n-graphs` pools at random

Draws `n_graphs` rows of the pool index at random over all files at once, so every region keeps its share of pools on
average. With `n_graphs = None` all pools are kept.

In [13]:
def sample_pools(pool_index, n_graphs, seed):
    if n_graphs is None or n_graphs >= pool_index.height:
        return pool_index
    rng = np.random.default_rng(seed)
    positions = rng.choice(pool_index.height, size=n_graphs, replace=False)
    return pool_index[positions]

**Line by line** with `n_graphs = 15`, `seed = 42`.

In [14]:
# the inputs
pool_index, n_graphs, seed = train_index_all, 15, 42

# if n_graphs is None or n_graphs >= pool_index.height: return everything (no sample wanted, or not enough pools)
print("pool_index.height =", pool_index.height)
print("n_graphs is None:", n_graphs is None, "| n_graphs >= height:", n_graphs >= pool_index.height,
      "-> returns all:", n_graphs is None or n_graphs >= pool_index.height)

pool_index.height = 34
n_graphs is None: False | n_graphs >= height: False -> returns all: False


In [15]:
# rng = np.random.default_rng(seed): a local random generator. Same seed -> same sequence of random numbers,
# so the same --seed always gives the same sample. It does not touch numpy's global generator.
rng = np.random.default_rng(seed)
show("rng", rng)

rng: Generator
Generator(PCG64) 



In [16]:
# positions = rng.choice(height, size=n_graphs, replace=False): draws n_graphs different row numbers.
# replace=False -> no row twice, so no pool appears twice in the sample.
positions = rng.choice(pool_index.height, size=n_graphs, replace=False)
show("positions", positions)

positions: ndarray | dtype int64 | shape (15,)
[ 1 28 14 25  5 24  2 31 10 21 23 15 16 18 30] 



In [17]:
# pool_index[positions]: indexing a DataFrame with an integer array returns these rows, in this order
sampled = pool_index[positions]
show("sampled", sampled)
print("check, same as the function:", sampled.equals(sample_pools(train_index_all, 15, 42)))
print("pools per file in the sample:", sampled.group_by("graph_file").len()["len"].to_list())
train_index = sampled

sampled: DataFrame | shape (15, 2)
shape: (15, 2)
┌──────────┬─────────────────────────────────┐
│ seed_pid ┆ graph_file                      │
│ ---      ┆ ---                             │
│ str      ┆ str                             │
╞══════════╪═════════════════════════════════╡
│ A1_1_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_6_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A2_2_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_3_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A1_5_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ …        ┆ …                               │
│ B1_1_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A2_3_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A2_4_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ A2_6_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_8_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
└──────────┴─────────────────────────────────┘ 

check, same as the function: True
pools per file in the sample: [6, 4, 5]


## 3. Loading the pools

### 3.1 `load_pools` — the points of the sampled pools into memory

Reads every file only once (the sampled pools are grouped by file), keeps only the rows of the sampled pools and turns
every pool into a dict of numpy arrays. Pools smaller than `min_graph_size` are dropped.

In [18]:
def load_pools(sampled_index, use_gse, min_graph_size):
    cols = ["latitude", "longitude", "mean_velocity", "incidence_angle", "seed_pid"]
    gse_cols = [f"gse_{i}" for i in range(64)]
    if use_gse:
        cols += gse_cols
    pools = []
    for (graph_file,), rows in sampled_index.group_by("graph_file"):
        df = pl.read_parquet(graph_file, columns=cols)
        df = df.filter(pl.col("seed_pid").is_in(rows["seed_pid"].implode()))
        pools_in_file = df.partition_by("seed_pid", as_dict=True)
        for pool_df in pools_in_file.values():
            if pool_df.height < min_graph_size:
                continue
            pool = {"lat": pool_df["latitude"].to_numpy(), "lon": pool_df["longitude"].to_numpy(),
                    "vel": pool_df["mean_velocity"].to_numpy().astype(np.float32),
                    "inc": pool_df["incidence_angle"].to_numpy().astype(np.float32),
                    "region": Path(graph_file).parent.parent.name, "orbit": Path(graph_file).parent.name}
            if use_gse:
                pool["gse"] = pool_df.select(gse_cols).to_numpy()
            pools.append(pool)
    return pools

**Line by line** with `use_gse = True`, `min_graph_size = 7`.

In [26]:
# the inputs
sampled_index, use_gse, min_graph_size = train_index, True, 7

# the columns to read: only what the graphs need (easting/northing, pid, seed flag ... are never read)
cols = ["latitude", "longitude", "mean_velocity", "incidence_angle", "seed_pid"]
# the 64 GSE column names, built with a list comprehension: gse_0 ... gse_63
gse_cols = [f"gse_{i}" for i in range(64)]
print("cols:", cols)
print("gse_cols:", gse_cols[:3], "...", gse_cols[-1], "| count:", len(gse_cols))

cols: ['latitude', 'longitude', 'mean_velocity', 'incidence_angle', 'seed_pid']
gse_cols: ['gse_0', 'gse_1', 'gse_2'] ... gse_63 | count: 64


In [27]:
# cols += gse_cols: appends the 64 names to the list (only with GSE; without it the GSE columns are never read)
if use_gse:
    cols += gse_cols
print("number of columns to read:", len(cols))
# the list that collects the finished pools
pools = []

number of columns to read: 69


In [29]:
# for (graph_file,), rows in sampled_index.group_by("graph_file"):
# group_by gives one group per file: the key (a tuple, because one can group by several columns -> unpacked with
# (graph_file,)) and the rows of the sample that belong to this file
groups = list(sampled_index.group_by("graph_file"))
print(groups)
print("number of groups (= files with sampled pools):", len(groups))
(graph_file,), rows = groups[0]
print("graph_file:", graph_file)
show("rows", rows)

[(('/dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionB/ascending/burstB1_GRAPH.parquet',), shape: (6, 2)
┌──────────┬─────────────────────────────────┐
│ seed_pid ┆ graph_file                      │
│ ---      ┆ ---                             │
│ str      ┆ str                             │
╞══════════╪═════════════════════════════════╡
│ B1_6_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_3_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_2_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_9_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_1_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
│ B1_8_0   ┆ /dss/dsstbyfs02/scratch/0C/di5… │
└──────────┴─────────────────────────────────┘), (('/dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionA/ascending/burstA1_GRAPH.parquet',), shape: (4, 2)
┌──────────┬─────────────────────────────────┐
│ seed_pid ┆ graph_file                      │
│ ---      ┆ ---                             │
│ str      ┆ str                      

In [30]:
# df = pl.read_parquet(graph_file, columns=cols): reads the file, but only these columns (parquet is stored column by
# column, so unread columns cost nothing)
df = pl.read_parquet(graph_file, columns=cols)
show("df", df.select(cols[:5]))

df: DataFrame | shape (194, 5)
shape: (194, 5)
┌───────────┬───────────┬───────────────┬─────────────────┬──────────┐
│ latitude  ┆ longitude ┆ mean_velocity ┆ incidence_angle ┆ seed_pid │
│ ---       ┆ ---       ┆ ---           ┆ ---             ┆ ---      │
│ f64       ┆ f64       ┆ f64           ┆ f64             ┆ str      │
╞═══════════╪═══════════╪═══════════════╪═════════════════╪══════════╡
│ 48.129138 ┆ 11.463878 ┆ -2.705161     ┆ 41.78941        ┆ B1_0_0   │
│ 48.125586 ┆ 11.466834 ┆ 0.002072      ┆ 35.396916       ┆ B1_0_0   │
│ 48.134343 ┆ 11.461368 ┆ -1.190126     ┆ 34.291424       ┆ B1_0_0   │
│ 48.131673 ┆ 11.46872  ┆ -0.638223     ┆ 37.610613       ┆ B1_0_0   │
│ 48.133429 ┆ 11.460762 ┆ -1.117707     ┆ 34.049577       ┆ B1_0_0   │
│ …         ┆ …         ┆ …             ┆ …               ┆ …        │
│ 48.132659 ┆ 11.465992 ┆ -0.632446     ┆ 37.844031       ┆ B1_11_0  │
│ 48.130101 ┆ 11.4644   ┆ -0.73924      ┆ 36.442005       ┆ B1_11_0  │
│ 48.134389 ┆ 11.473639 ┆ 0.04

In [31]:
# rows["seed_pid"].implode(): the sampled pool ids of this file as ONE list (a Series with one element that is a list).
# is_in(...) then checks for every row whether its seed_pid is in that list. implode makes it explicit that the whole
# list is one collection to compare against (polars warns without it)
wanted = rows["seed_pid"].implode()
show("wanted", wanted)
mask = df.select(pl.col("seed_pid").is_in(wanted))["seed_pid"]
print("True = row belongs to a sampled pool:", mask.sum(), "of", len(mask), "rows")
print(df.with_columns(mask.alias("is_in")).select("seed_pid", "is_in").unique(maintain_order=True))

wanted: Series | dtype List(String) | shape (1,)
shape: (1,)
Series: 'seed_pid' [list[str]]
[
	["B1_6_0", "B1_3_0", … "B1_8_0"]
] 

True = row belongs to a sampled pool: 95 of 194 rows
shape: (12, 2)
┌──────────┬───────┐
│ seed_pid ┆ is_in │
│ ---      ┆ ---   │
│ str      ┆ bool  │
╞══════════╪═══════╡
│ B1_0_0   ┆ false │
│ B1_1_0   ┆ true  │
│ B1_2_0   ┆ true  │
│ B1_3_0   ┆ true  │
│ B1_4_0   ┆ false │
│ …        ┆ …     │
│ B1_7_0   ┆ false │
│ B1_8_0   ┆ true  │
│ B1_9_0   ┆ true  │
│ B1_10_0  ┆ false │
│ B1_11_0  ┆ false │
└──────────┴───────┘


In [32]:
# df = df.filter(...): keep only those rows -> only the points of the sampled pools of this file
df = df.filter(pl.col("seed_pid").is_in(rows["seed_pid"].implode()))
print("rows after the filter:", df.height)
print("pools left:", sorted(df["seed_pid"].unique().to_list()))
print("sampled pools of this file:", sorted(rows["seed_pid"].to_list()))

rows after the filter: 95
pools left: ['B1_1_0', 'B1_2_0', 'B1_3_0', 'B1_6_0', 'B1_8_0', 'B1_9_0']
sampled pools of this file: ['B1_1_0', 'B1_2_0', 'B1_3_0', 'B1_6_0', 'B1_8_0', 'B1_9_0']


In [33]:
# pools_in_file = df.partition_by("seed_pid", as_dict=True): splits the table into one DataFrame per pool.
# as_dict=True -> a dict {(seed_pid,): DataFrame}
pools_in_file = df.partition_by("seed_pid", as_dict=True)
print("type:", type(pools_in_file).__name__)
for key, part in pools_in_file.items():
    print(f"  key {key} -> DataFrame with {part.height} rows")

type: dict
  key ('B1_1_0',) -> DataFrame with 19 rows
  key ('B1_2_0',) -> DataFrame with 22 rows
  key ('B1_3_0',) -> DataFrame with 7 rows
  key ('B1_6_0',) -> DataFrame with 18 rows
  key ('B1_8_0',) -> DataFrame with 13 rows
  key ('B1_9_0',) -> DataFrame with 16 rows


In [34]:
# for pool_df in pools_in_file.values(): if pool_df.height < min_graph_size: continue
# pools with fewer points than the minimum graph size are skipped (continue = go on with the next pool)
for pool_df in pools_in_file.values():
    print(f"{pool_df['seed_pid'][0]}: {pool_df.height} points ->", "skipped" if pool_df.height < min_graph_size else "kept")
pool_df = next(p for p in pools_in_file.values() if p.height >= min_graph_size)   # the first kept pool, for the next cells

B1_1_0: 19 points -> kept
B1_2_0: 22 points -> kept
B1_3_0: 7 points -> kept
B1_6_0: 18 points -> kept
B1_8_0: 13 points -> kept
B1_9_0: 16 points -> kept


In [35]:
# the pool dict, field by field. .to_numpy() turns a polars column into a numpy array
lat_ = pool_df["latitude"].to_numpy()
show("lat (float64: full precision, needed for the geodesics)", lat_, precision=6)
vel_ = pool_df["mean_velocity"].to_numpy().astype(np.float32)
show("vel (.astype(np.float32): half the memory, the precision the model uses anyway)", vel_)

lat (float64: full precision, needed for the geodesics): ndarray | dtype float64 | shape (19,)
[48.119367 48.122641 48.122756 48.116683 48.121963 48.123072 48.120767 48.116028 48.11174  48.124454 48.117036 48.117527 48.117472
 48.112257 48.120425 48.117876 48.129428 48.119965 48.117529] 

vel (.astype(np.float32): half the memory, the precision the model uses anyway): ndarray | dtype float32 | shape (19,)
[-0.6813  0.9278 -1.0439 -2.2143  0.7456 -3.7674 -0.1344 -1.2835 -7.4577 -0.3557 -1.1798 -1.2112 -0.3438 -1.1794 -1.0692 -2.1755 -0.7208
 -1.2441 -0.5776] 



In [36]:
# region and orbit come from the folder names of the file: data/<set>/<location>/<orbit>/<burst>_GRAPH.parquet
p = Path(graph_file)
print("Path(graph_file).parent             :", p.parent)
print("Path(graph_file).parent.name        :", p.parent.name, "  <- orbit")
print("Path(graph_file).parent.parent.name :", p.parent.parent.name, "  <- region")

Path(graph_file).parent             : /dss/dsstbyfs02/scratch/0C/di54haf/di54haf/tmpstg3tafn/data/train/RegionB/ascending
Path(graph_file).parent.name        : ascending   <- orbit
Path(graph_file).parent.parent.name : RegionB   <- region


In [37]:
# pool["gse"] = pool_df.select(gse_cols).to_numpy(): the 64 GSE columns as one matrix, one row per point.
# They stay int8 (1 byte per value) and are only de-quantised when a graph is built (section 4.4)
gse_ = pool_df.select(gse_cols).to_numpy()
show("gse", gse_[:3, :8])
print("full shape:", gse_.shape)

gse: ndarray | dtype int8 | shape (3, 8)
[[-10  33 -30   9 -13  18 -42 -37]
 [ 49  55  61  29  41 -34  34  28]
 [ 22  53 -41 -44  46  17 -31  26]] 

full shape: (19, 64)


In [38]:
# the whole function: all files, all pools
USE_GSE, MIN_GRAPH_SIZE = True, 7
train_pools = load_pools(train_index, USE_GSE, MIN_GRAPH_SIZE)
print("sampled:", train_index.height, "| kept (>= min_graph_size):", len(train_pools))
pool = train_pools[0]
print("one pool is a", type(pool).__name__, "with keys", list(pool))
for k, v in pool.items():
    print(f"  {k:7s}", type(v).__name__, getattr(v, "dtype", ""), getattr(v, "shape", v))

sampled: 15 | kept (>= min_graph_size): 14
one pool is a dict with keys ['lat', 'lon', 'vel', 'inc', 'region', 'orbit', 'gse']
  lat     ndarray float64 (19,)
  lon     ndarray float64 (19,)
  vel     ndarray float32 (19,)
  inc     ndarray float32 (19,)
  region  str  RegionB
  orbit   str  ascending
  gse     ndarray int8 (19, 64)


### 3.2 `load_norm_stats` — mean and std of velocity and incidence

In s04 the stats come from `assets/egms_norm_stats.json`, written by s02 from the train regions. Here we write a toy
json in the same format (computed from the toy train pools, plus one extra column that is not needed) and read it with
the same code. `EGMS_NORM_STATS_PATH` and `EGMS_NORM_COLS` are module constants of s02 (`s02.…` in s04).

In [39]:
EGMS_NORM_COLS = ["mean_velocity", "incidence_angle"]
EGMS_NORM_STATS_PATH = Path(tempfile.mkdtemp()) / "egms_norm_stats.json"
all_vel = np.concatenate([p["vel"] for p in train_pools]); all_inc = np.concatenate([p["inc"] for p in train_pools])
EGMS_NORM_STATS_PATH.write_text(json.dumps({
    "mean_velocity": {"mean": float(all_vel.mean()), "std": float(all_vel.std())},
    "incidence_angle": {"mean": float(all_inc.mean()), "std": float(all_inc.std())},
    "temporal_coherence": {"mean": 0.8, "std": 0.1}}))

def load_norm_stats():
    with open(EGMS_NORM_STATS_PATH) as f:
        stats = json.load(f)
    return {c: stats[c] for c in EGMS_NORM_COLS}

In [40]:
# with open(path) as f: opens the file; "with" closes it again at the end of the block, also after an error
# json.load(f): parses the json text into python dicts
with open(EGMS_NORM_STATS_PATH) as f:
    raw_stats = json.load(f)
show("raw_stats", raw_stats)

raw_stats: dict
{'mean_velocity': {'mean': -6.550420761108398, 'std': 5.342747688293457}, 'incidence_angle': {'mean': 38.40049362182617, 'std': 3.1231307983398438}, 'temporal_coherence': {'mean': 0.8, 'std': 0.1}} 



In [41]:
# {c: stats[c] for c in EGMS_NORM_COLS}: a dict comprehension that keeps only the columns the model normalizes
stats = {c: raw_stats[c] for c in EGMS_NORM_COLS}
print(json.dumps(stats, indent=2))
print("check, same as the function:", stats == load_norm_stats())

{
  "mean_velocity": {
    "mean": -6.550420761108398,
    "std": 5.342747688293457
  },
  "incidence_angle": {
    "mean": 38.40049362182617,
    "std": 3.1231307983398438
  }
}
check, same as the function: True


## 4. One graph by hand — what `PoolDataset.__getitem__` does

The core of s04: from one pool a **random subset** is drawn, one point of it becomes the **query** (its velocity is
hidden), every point is connected to its **k nearest neighbours**, and the node and edge features are built.

### 4.1 `draw_subset` — which points, which query

In [42]:
def draw_subset(n, subset_min, subset_max, min_graph_size, rng):
    share = rng.uniform(subset_min, subset_max)
    m = min(n, max(min_graph_size, round(share * n)))
    idx = rng.choice(n, size=m, replace=False)
    q = rng.integers(m)
    return idx, q

**Line by line** on `pool = train_pools[0]`, subset share 0.6–0.9.

In [43]:
# the inputs: n = number of points of the pool
n = len(pool["vel"])
subset_min, subset_max, min_graph_size = 0.6, 0.9, MIN_GRAPH_SIZE
rng = np.random.default_rng(1)       # in PoolDataset this generator is seeded per pool (and epoch), see section 5
print("n =", n)

n = 19


In [44]:
# share = rng.uniform(subset_min, subset_max): a random share of the pool, uniformly between 60 and 90 %
share = rng.uniform(subset_min, subset_max)
print("share =", share)

share = 0.753546487410077


In [45]:
# m = min(n, max(min_graph_size, round(share * n))): the subset size, built from the inside out
print("share * n                         =", share * n)
print("round(share * n)                  =", round(share * n), " <- a whole number of points")
print("max(min_graph_size, ...)          =", max(min_graph_size, round(share * n)), " <- never below the minimum graph size")
m = min(n, max(min_graph_size, round(share * n)))
print("min(n, ...)                       =", m, " <- never more points than the pool has")

share * n                         = 14.317383260791463
round(share * n)                  = 14  <- a whole number of points
max(min_graph_size, ...)          = 14  <- never below the minimum graph size
min(n, ...)                       = 14  <- never more points than the pool has


In [46]:
# idx = rng.choice(n, size=m, replace=False): m different positions out of 0 ... n-1 -> which points are in the graph.
# The order is random, so the first node of the graph is not necessarily the seed
idx = rng.choice(n, size=m, replace=False)
show("idx (positions in the pool)", idx)

idx (positions in the pool): ndarray | dtype int64 | shape (14,)
[17 15  0 10 16  8  1  7  2 13  4  6 12 14] 



In [47]:
# q = rng.integers(m): a random position WITHIN idx (0 ... m-1) -> which node of the graph is the query
q = rng.integers(m)
print("q =", q, "-> node", q, "of the graph = point", idx[q], "of the pool")

q = 4 -> node 4 of the graph = point 16 of the pool


In [48]:
# check: a fresh generator with the same seed gives exactly the same subset and query
idx_f, q_f = draw_subset(n, 0.6, 0.9, MIN_GRAPH_SIZE, np.random.default_rng(1))
print("check, same as the function:", np.array_equal(idx, idx_f) and q == q_f)

check, same as the function: True


### 4.2 `pairwise_geodesic` — all distances and bearings

Computes the geodesic distance and bearing between **all pairs** of the subset's points in one `GEOD.inv` call.

In [49]:
def pairwise_geodesic(lat, lon):
    m = len(lat)
    lat1, lon1 = np.repeat(lat, m), np.repeat(lon, m)
    lat2, lon2 = np.tile(lat, m), np.tile(lon, m)
    az12, _, d = GEOD.inv(lon1, lat1, lon2, lat2)
    return d.reshape(m, m), az12.reshape(m, m)

**Line by line.** First on 3 points, so every array fits on one line, then on the real subset.

In [50]:
# the inputs: the coordinates of the subset's points (pool["lat"][idx] = the latitudes at the positions idx)
lat, lon = pool["lat"][idx], pool["lon"][idx]
show("lat", lat, precision=6)
show("lon", lon, precision=6)
lat3, lon3 = lat[:3], lon[:3]    # the first 3 points for the small example
m3 = len(lat3)
print("m3 =", m3)

lat: ndarray | dtype float64 | shape (14,)
[48.119965 48.117876 48.119367 48.117036 48.129428 48.11174  48.122641 48.116028 48.122756 48.112257 48.121963 48.120767 48.117472
 48.120425] 

lon: ndarray | dtype float64 | shape (14,)
[11.464525 11.47159  11.466812 11.46714  11.470538 11.485065 11.468987 11.474196 11.46127  11.470008 11.463755 11.459372 11.470193
 11.46268 ] 

m3 = 3


In [51]:
# np.repeat(lat, m): every value m times in a row -> the FROM point of every pair
# np.tile(lat, m):   the whole array m times after each other -> the TO point of every pair
# side by side they give every (i, j) combination exactly once: (0,0) (0,1) (0,2) (1,0) (1,1) ...
lat1, lon1 = np.repeat(lat3, m3), np.repeat(lon3, m3)
lat2, lon2 = np.tile(lat3, m3), np.tile(lon3, m3)
print("np.repeat([0, 1, 2], 3):", np.repeat([0, 1, 2], 3))
print("np.tile([0, 1, 2], 3):  ", np.tile([0, 1, 2], 3))
show("lat1 (from)", lat1, precision=6)
show("lat2 (to)", lat2, precision=6)

np.repeat([0, 1, 2], 3): [0 0 0 1 1 1 2 2 2]
np.tile([0, 1, 2], 3):   [0 1 2 0 1 2 0 1 2]
lat1 (from): ndarray | dtype float64 | shape (9,)
[48.119965 48.119965 48.119965 48.117876 48.117876 48.117876 48.119367 48.119367 48.119367] 

lat2 (to): ndarray | dtype float64 | shape (9,)
[48.119965 48.117876 48.119367 48.119965 48.117876 48.119367 48.119965 48.117876 48.119367] 



In [52]:
# az12, az21, d = GEOD.inv(lon1, lat1, lon2, lat2): one call for all m*m pairs (pyproj takes lon BEFORE lat).
# az12 = bearing from point 1 to point 2 (degrees, -180 ... 180, 0 = north, clockwise)
# az21 = the back bearing (2 -> 1), not needed: it is az12 of the reversed pair, i.e. the transposed matrix
# d    = distance in metres along the ellipsoid
az12, az21, d = GEOD.inv(lon1, lat1, lon2, lat2)
show("az12", az12, precision=1)
show("d", d, precision=1)

az12: ndarray | dtype float64 | shape (9,)
[180.  113.8 111.4 -66.2 180.  -65.  -68.6 115.  180. ] 

d: ndarray | dtype float64 | shape (9,)
[  0.  575.  182.8 575.    0.  392.5 182.8 392.5   0. ] 



In [53]:
# d.reshape(m, m): the flat list of m*m values back into a matrix. Because of repeat/tile the order is row by row:
# row i = from point i, column j = to point j. The diagonal (a point with itself) is 0
show("dist (3 x 3)", d.reshape(m3, m3), precision=1)
show("az (3 x 3)", az12.reshape(m3, m3), precision=1)

dist (3 x 3): ndarray | dtype float64 | shape (3, 3)
[[  0.  575.  182.8]
 [575.    0.  392.5]
 [182.8 392.5   0. ]] 

az (3 x 3): ndarray | dtype float64 | shape (3, 3)
[[180.  113.8 111.4]
 [-66.2 180.  -65. ]
 [-68.6 115.  180. ]] 



In [54]:
# the same on the whole subset, with the function
dist, az = pairwise_geodesic(lat, lon)
show("dist, first 5 x 5 (m)", dist[:5, :5], precision=0)
show("az, first 5 x 5 (deg), row i -> column j", az[:5, :5], precision=0)
print("distance is symmetric:", np.allclose(dist, dist.T))
print("bearing i->j and j->i differ by ~180 deg:", round(az[0, 1]), round(az[1, 0]))
# the diagonal of az (a point to itself) has no meaning (pyproj returns 180); it is never used, because knn_edges
# never connects a node to itself (the self loops GATv2Conv adds in section 7 get averaged edge features instead)

dist, first 5 x 5 (m): ndarray | dtype float64 | shape (5, 5)
[[   0.  575.  183.  380. 1143.]
 [ 575.    0.  392.  344. 1287.]
 [ 183.  392.    0.  260. 1153.]
 [ 380.  344.  260.    0. 1401.]
 [1143. 1287. 1153. 1401.    0.]] 

az, first 5 x 5 (deg), row i -> column j: ndarray | dtype float64 | shape (5, 5)
[[ 180.  114.  111.  149.   23.]
 [ -66.  180.  -65. -106.   -3.]
 [ -69.  115.  180.  175.   14.]
 [ -31.   74.   -5.  180.   10.]
 [-157.  177. -166. -170.  180.]] 

distance is symmetric: True
bearing i->j and j->i differ by ~180 deg: 114 -66


### 4.3 `knn_edges` — who is connected to whom

Every node is connected to its `k` nearest nodes; then every edge is added in the other direction as well.

In [55]:
def knn_edges(dist, k):
    m = len(dist)
    dist = dist.copy()
    np.fill_diagonal(dist, np.inf)
    neighbours = np.argsort(dist, axis=1)[:, :k]
    target = np.repeat(np.arange(m), k)
    source = neighbours.reshape(-1)
    edge_index = torch.tensor(np.stack([source, target]), dtype=torch.long)
    edge_index = to_undirected(edge_index)
    return edge_index

**Line by line** with `k = 3`.

In [56]:
# the inputs
K = 3
k = K
m = len(dist)
print("m =", m)

m = 14


In [57]:
# dist = dist.copy(): work on a copy. fill_diagonal changes the array IN PLACE, and the caller still needs the
# original distances for the edge features
dist_c = dist.copy()
print("same object?", dist_c is dist, "| same values?", np.array_equal(dist_c, dist))

same object? False | same values? True


In [58]:
# np.fill_diagonal(dist, np.inf): the distance of a node to itself is 0, so every node would be its own nearest
# neighbour. Infinity puts the node itself at the very end of the sorting
np.fill_diagonal(dist_c, np.inf)
show("dist_c, first 4 x 4", dist_c[:4, :4], precision=0)
print("the original is unchanged, diagonal:", np.diag(dist)[:4])

dist_c, first 4 x 4: ndarray | dtype float64 | shape (4, 4)
[[ inf 575. 183. 380.]
 [575.  inf 392. 344.]
 [183. 392.  inf 260.]
 [380. 344. 260.  inf]] 

the original is unchanged, diagonal: [0. 0. 0. 0.]


In [59]:
# np.argsort(dist, axis=1): per row the column numbers sorted by distance (nearest first). axis=1 = along the row
order = np.argsort(dist_c, axis=1)
show("argsort, first 3 rows", order[:3])
print("row 0: node", order[0, 0], "is nearest to node 0 at", round(dist[0, order[0, 0]]), "m")

argsort, first 3 rows: ndarray | dtype int64 | shape (3, 14)
[[13  2 10  3  8 11  6 12  1  7  9  4  5  0]
 [12  7  3  2  6  0  9 13 10  8 11  5  4  1]
 [ 0  3 12 13 10  1  6  8 11  7  9  4  5  2]] 

row 0: node 13 is nearest to node 0 at 147 m


In [60]:
# [:, :k]: all rows, the first k columns -> the k nearest neighbours of every node
neighbours = order[:, :k]
show("neighbours (row a = the k nodes closest to a)", neighbours[:5])

neighbours (row a = the k nodes closest to a): ndarray | dtype int64 | shape (5, 3)
[[13  2 10]
 [12  7  3]
 [ 0  3 12]
 [12  2  1]
 [ 6 10  8]] 



In [ ]:
# target = np.repeat(np.arange(m), k): node 0 k times, node 1 k times, ... -> the RECEIVING node of every edge
# source = neighbours.reshape(-1): the neighbour table flattened row by row -> the SENDING node of every edge
target = np.repeat(np.arange(m), k)
source = neighbours.reshape(-1)
show("target", target[:9])
show("source", source[:9])
print("so the first edges are:", [f"{s}->{t}" for s, t in zip(source[:6], target[:6])])


NameError: name 'np' is not defined

In [62]:
# np.stack([source, target]): the two rows on top of each other -> shape (2, n_edges), PyG's edge_index convention:
# row 0 = source, row 1 = target; messages flow from source to target.
# torch.tensor(..., dtype=torch.long): PyG needs the indices as 64-bit integers
directed = torch.tensor(np.stack([source, target]), dtype=torch.long)
show("directed", directed[:, :9])

directed: Tensor | dtype torch.int64 | shape (2, 9)
tensor([[13,  2, 10, 12,  7,  3,  0,  3, 12],
        [ 0,  0,  0,  1,  1,  1,  2,  2,  2]]) 



In [63]:
# to_undirected(edge_index): adds every edge in the other direction and removes duplicates.
# If b is among the 3 nearest of a, but a is not among the 3 nearest of b, the edge a->b is new
edge_index = to_undirected(directed)
print("edges before:", directed.shape[1], "| after:", edge_index.shape[1])
before = set(map(tuple, directed.T.tolist())); after = set(map(tuple, edge_index.T.tolist()))
print("examples of added edges (source, target):", sorted(after - before)[:6])
print("in-degree per node (incoming edges):", torch.bincount(edge_index[1]).tolist(), "-> at least k everywhere")
print("check, same as the function:", torch.equal(edge_index, knn_edges(dist, K)))

edges before: 42 | after: 56
examples of added edges (source, target): [(4, 6), (4, 8), (4, 10), (5, 1), (5, 7), (5, 9)]
in-degree per node (incoming edges): [4, 4, 4, 4, 3, 3, 4, 4, 4, 4, 6, 3, 5, 4] -> at least k everywhere
check, same as the function: True


### 4.4 `dequantise_gse` — int8 back to the unit-length embedding

AlphaEarth stores every value as int8 (−127 … 127). The values were square-root scaled before rounding, which gives more
resolution near 0; de-quantising squares them again and puts the sign back: `x = (v / 127.5)² · sign(v)`.

In [64]:
def dequantise_gse(v):
    v = v.astype(np.float32)
    x = (v / 127.5) ** 2 * np.sign(v)
    return x

In [65]:
# the input: the raw int8 values of the subset's points (2 points x 6 of the 64 dimensions shown)
v = pool["gse"][idx]
show("v (raw)", v[:2, :6])

v (raw): ndarray | dtype int8 | shape (2, 6)
[[ 25 -34 -50  38 -31 -16]
 [ 43  37  41 -38  47  17]] 



In [66]:
# v.astype(np.float32): int8 can only hold -128 ... 127, so arithmetic must happen in float
print("why: in int8, 100 + 100 =", (np.array([100], dtype=np.int8) + np.array([100], dtype=np.int8))[0], "(overflow)")
v = v.astype(np.float32)
show("v (float32)", v[:2, :6])

why: in int8, 100 + 100 = -56 (overflow)
v (float32): ndarray | dtype float32 | shape (2, 6)
[[ 25. -34. -50.  38. -31. -16.]
 [ 43.  37.  41. -38.  47.  17.]] 



In [67]:
# the formula piece by piece
print("v / 127.5           :", (v / 127.5)[0, :6], " <- scaled to -1 ... 1")
print("(v / 127.5) ** 2    :", ((v / 127.5) ** 2)[0, :6], " <- squared, the sign is lost")
print("np.sign(v)          :", np.sign(v)[0, :6], " <- -1, 0 or +1")
x_gse = (v / 127.5) ** 2 * np.sign(v)
print("(...) ** 2 * sign(v):", x_gse[0, :6], " <- the sign back")
print("\nvector length per point (should be ~1):", np.linalg.norm(x_gse, axis=1)[:5])
print("check, same as the function:", np.array_equal(x_gse, dequantise_gse(pool["gse"][idx])))

v / 127.5           : [ 0.1961 -0.2667 -0.3922  0.298  -0.2431 -0.1255]  <- scaled to -1 ... 1
(v / 127.5) ** 2    : [0.0384 0.0711 0.1538 0.0888 0.0591 0.0157]  <- squared, the sign is lost
np.sign(v)          : [ 1. -1. -1.  1. -1. -1.]  <- -1, 0 or +1
(...) ** 2 * sign(v): [ 0.0384 -0.0711 -0.1538  0.0888 -0.0591 -0.0157]  <- the sign back

vector length per point (should be ~1): [0.9999 1.0044 0.9994 1.0027 1.0002]
check, same as the function: True


### 4.5 `build_node_features` — the node matrix `x`

One row per node: `[normalized velocity (0 at the query), is_query, normalized incidence]`, plus the 64 GSE values.

In [69]:
def build_node_features(pool, idx, q, stats, use_gse):
    m = len(idx)
    v_stats, i_stats = stats["mean_velocity"], stats["incidence_angle"]
    vel = (pool["vel"][idx] - v_stats["mean"]) / v_stats["std"]
    vel[q] = 0.0
    is_query = np.zeros(m, dtype=np.float32)
    is_query[q] = 1.0
    inc = (pool["inc"][idx] - i_stats["mean"]) / i_stats["std"]
    gse = dequantise_gse(pool["gse"][idx]) if use_gse else None
    cols = [vel, is_query, inc]
    x = np.column_stack(cols + ([gse] if use_gse else []))
    return torch.tensor(x, dtype=torch.float32)

In [70]:
# m = len(idx); the stats of the two normalized columns
m = len(idx)
v_stats, i_stats = stats["mean_velocity"], stats["incidence_angle"]
print("m =", m, "| v_stats:", v_stats, "| i_stats:", i_stats)

m = 14 | v_stats: {'mean': -6.550420761108398, 'std': 5.342747688293457} | i_stats: {'mean': 38.40049362182617, 'std': 3.1231307983398438}


In [71]:
# vel = (pool["vel"][idx] - mean) / std: standardization -> mean 0, std 1 over the train regions.
# The subtraction creates a NEW array, so the pool itself is not changed by the next line
print("raw velocity (mm/yr):", pool["vel"][idx][:6])
vel = (pool["vel"][idx] - v_stats["mean"]) / v_stats["std"]
print("normalized          :", vel[:6])

raw velocity (mm/yr): [-1.2441 -2.1755 -0.6813 -1.1798 -0.7208 -7.4577]
normalized          : [ 0.9932  0.8188  1.0985  1.0052  1.0911 -0.1698]


In [72]:
# vel[q] = 0.0: the query's velocity is hidden -> this is the masked target. The model only sees 0 there
true_q = vel[q]
vel[q] = 0.0
print(f"node q = {q}: normalized value {true_q:.4f} replaced by {vel[q]}")
print("the pool still has the true value:", pool["vel"][idx][q], "mm/yr")

node q = 4: normalized value 1.0911 replaced by 0.0
the pool still has the true value: -0.72075975 mm/yr


In [73]:
# is_query: 0 everywhere, 1 at the query. Without this flag the model could not tell the hidden query (vel = 0) from a
# point whose velocity really is 0
is_query = np.zeros(m, dtype=np.float32)
is_query[q] = 1.0
show("is_query", is_query)

is_query: ndarray | dtype float32 | shape (14,)
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.] 



In [74]:
# inc: the incidence angle, normalized in the same way
inc = (pool["inc"][idx] - i_stats["mean"]) / i_stats["std"]
print("raw (deg):", pool["inc"][idx][:6], "\nnormalized:", inc[:6])

raw (deg): [35.5643 37.1973 34.5153 42.7704 37.1579 34.1899] 
normalized: [-0.9081 -0.3853 -1.244   1.3992 -0.3979 -1.3482]


In [75]:
# gse = dequantise_gse(...) if use_gse else None: a one-line if/else (conditional expression)
use_gse = True
gse = dequantise_gse(pool["gse"][idx]) if use_gse else None
print("gse shape:", None if gse is None else gse.shape)

gse shape: (14, 64)


In [76]:
# x = np.column_stack(cols + ([gse] if use_gse else [])): column_stack puts 1-d arrays side by side as columns and
# appends the 64 columns of the GSE matrix. [gse] if use_gse else [] adds the matrix to the list only with GSE
cols = [vel, is_query, inc]
x_np = np.column_stack(cols + ([gse] if use_gse else []))
x_np_no_gse = np.column_stack(cols)
print("with GSE:", x_np.shape, "| without:", x_np_no_gse.shape)
print("columns [vel, is_query, inc] (the query row has vel = 0 and is_query = 1):")
print(x_np_no_gse)

with GSE: (14, 67) | without: (14, 3)
columns [vel, is_query, inc] (the query row has vel = 0 and is_query = 1):
[[ 0.9932  0.     -0.9081]
 [ 0.8188  0.     -0.3853]
 [ 1.0985  0.     -1.244 ]
 [ 1.0052  0.      1.3992]
 [ 0.      1.     -0.3979]
 [-0.1698  0.     -1.3482]
 [ 1.3997  0.     -1.3206]
 [ 0.9858  0.     -0.1835]
 [ 1.0306  0.      0.1116]
 [ 1.0053  0.      0.3969]
 [ 1.3656  0.     -0.862 ]
 [ 1.2009  0.     -0.6866]
 [ 1.1617  0.     -0.59  ]
 [ 1.0259  0.      0.4645]]


In [77]:
# torch.tensor(x, dtype=torch.float32): the model works with float32 torch tensors
x = torch.tensor(x_np, dtype=torch.float32)
show("x", x[:3, :6])
print("check, same as the function:", torch.equal(x, build_node_features(pool, idx, q, stats, True)))

x: Tensor | dtype torch.float32 | shape (3, 6)
tensor([[ 0.9932,  0.0000, -0.9081,  0.0384, -0.0711, -0.1538],
        [ 0.8188,  0.0000, -0.3853,  0.1137,  0.0842,  0.1034],
        [ 1.0985,  0.0000, -1.2440, -0.0062,  0.0670, -0.0554]]) 

check, same as the function: True


### 4.6 `build_edge_features` — one row per edge

`[log distance (normalized once the edge stats exist), sin(bearing), cos(bearing)]`, the bearing taken from the
receiving to the sending node.

In [78]:
def build_edge_features(dist, az, edge_index, stats):
    src, dst = edge_index.numpy()
    d = np.log1p(dist[src, dst])
    if "edge_dist" in stats:
        d = (d - stats["edge_dist"]["mean"]) / stats["edge_dist"]["std"]
    bearing = np.radians(az[dst, src])
    sin, cos = np.sin(bearing), np.cos(bearing)
    edge_attr = np.column_stack([d, sin, cos])
    return torch.tensor(edge_attr, dtype=torch.float32)

In [79]:
# src, dst = edge_index.numpy(): the (2, n_edges) array unpacked into its two rows: senders and receivers
src, dst = edge_index.numpy()
show("src", src[:8])
show("dst", dst[:8])

src: ndarray | dtype int64 | shape (8,)
[0 0 0 0 1 1 1 1] 

dst: ndarray | dtype int64 | shape (8,)
[ 2  6 10 13  3  5  7 12] 



In [80]:
# dist[src, dst]: fancy indexing with two index arrays -> element-wise pairs (src[0], dst[0]), (src[1], dst[1]), ...
# so one distance per edge, in the order of the edges
d_edge = dist[src, dst]
for e in range(4):
    print(f"edge {e}: {src[e]} -> {dst[e]}   dist[{src[e]}, {dst[e]}] = {dist[src[e], dst[e]]:.1f} m   (from the array: {d_edge[e]:.1f})")

edge 0: 0 -> 2   dist[0, 2] = 182.8 m   (from the array: 182.8)
edge 1: 0 -> 6   dist[0, 6] = 445.9 m   (from the array: 445.9)
edge 2: 0 -> 10   dist[0, 10] = 229.4 m   (from the array: 229.4)
edge 3: 0 -> 13   dist[0, 13] = 146.6 m   (from the array: 146.6)


In [81]:
# np.log1p(x) = log(1 + x): compresses the range (a few m in cities, km in rural areas) so that the difference
# between 10 and 20 m counts about as much as between 1 and 2 km. The +1 keeps log finite at distance 0
print("log1p of 0, 10, 100, 1000, 10000 m:", np.log1p([0, 10, 100, 1000, 10000]))
d = np.log1p(d_edge)
show("d", d[:6])

log1p of 0, 10, 100, 1000, 10000 m: [0.     2.3979 4.6151 6.9088 9.2104]
d: ndarray | dtype float64 | shape (6,)
[5.2138 6.1024 5.4398 4.9947 5.8443 7.102 ] 



In [82]:
# if "edge_dist" in stats: only once estimate_edge_stats has run (section 5.2) the log distance is standardized.
# Here the stats don't have it yet, so d stays the raw log distance
print('"edge_dist" in stats:', "edge_dist" in stats, "-> d stays raw for now")

"edge_dist" in stats: False -> d stays raw for now


In [83]:
# bearing = np.radians(az[dst, src]): note the order [dst, src] -> the bearing FROM the receiving node TO the sender,
# i.e. "where does the message come from, seen from the node that receives it". np.radians: sin/cos expect radians
bearing = np.radians(az[dst, src])
print("degrees:", az[dst, src][:5], "\nradians:", bearing[:5])

degrees: [ -68.6434 -131.8454  165.5209  110.4157   74.2456] 
radians: [-1.1981 -2.3011  2.8889  1.9271  1.2958]


In [84]:
# sin and cos: an angle as two numbers on the unit circle. 1 deg and 359 deg are almost the same direction, and their
# (sin, cos) are close too, while the raw numbers 1 and 359 are far apart
for deg in (1, 359, 90, -90):
    r = np.radians(deg)
    print(f"{deg:4d} deg -> sin {np.sin(r):6.3f}, cos {np.cos(r):6.3f}")
sin, cos = np.sin(bearing), np.cos(bearing)

   1 deg -> sin  0.017, cos  1.000
 359 deg -> sin -0.017, cos  1.000
  90 deg -> sin  1.000, cos  0.000
 -90 deg -> sin -1.000, cos  0.000


In [85]:
# np.column_stack([d, sin, cos]) -> (n_edges, 3), then a float32 tensor
edge_attr = torch.tensor(np.column_stack([d, sin, cos]), dtype=torch.float32)
show("edge_attr", edge_attr[:5])
print("sin^2 + cos^2 = 1:", torch.allclose(edge_attr[:, 1] ** 2 + edge_attr[:, 2] ** 2, torch.ones(len(edge_attr))))
print("check, same as the function:", torch.equal(edge_attr, build_edge_features(dist, az, edge_index, stats)))

edge_attr: Tensor | dtype torch.float32 | shape (5, 3)
tensor([[ 5.2138, -0.9313,  0.3642],
        [ 6.1024, -0.7449, -0.6671],
        [ 5.4398,  0.2500, -0.9682],
        [ 4.9947,  0.9372, -0.3488],
        [ 5.8443,  0.9624,  0.2715]]) 

sin^2 + cos^2 = 1: True
check, same as the function: True


### 4.7 Target, query mask and the finished `Data` object (the end of `__getitem__`)

In [86]:
# y: the query's TRUE velocity, normalized like the input. torch.tensor([...]) with the brackets gives shape (1,), so
# the DataLoader can concatenate the y of all graphs of a batch into one vector of shape (n_graphs,)
print("true velocity of the query:", pool["vel"][idx][q], "mm/yr")
y = torch.tensor([(pool["vel"][idx][q] - stats["mean_velocity"]["mean"]) / stats["mean_velocity"]["std"]], dtype=torch.float32)
show("y", y)

true velocity of the query: -0.72075975 mm/yr
y: Tensor | dtype torch.float32 | shape (1,)
tensor([1.0911]) 



In [87]:
# query_mask: a boolean vector, True only at the query. The model uses it to read its output at the query node
query_mask = torch.zeros(len(idx), dtype=torch.bool)
query_mask[q] = True
show("query_mask", query_mask)

query_mask: Tensor | dtype torch.bool | shape (14,)
tensor([False, False, False, False,  True, False, False, False, False, False,
        False, False, False, False]) 



In [88]:
# Data(...): PyG's container for one graph. Any keyword becomes an attribute (data.x, data.y, data.region, ...)
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, query_mask=query_mask,
            region=pool["region"], orbit=pool["orbit"])
print(data)
print("num_nodes:", data.num_nodes, "| num_edges:", data.num_edges)
print("the hidden value is NOT in x:", x[q, 0].item(), "(only in y)")

Data(x=[14, 67], edge_index=[2, 56], edge_attr=[56, 3], y=[1], query_mask=[14], region='RegionB', orbit='ascending')
num_nodes: 14 | num_edges: 56
the hidden value is NOT in x: 0.0 (only in y)


## 5. Packed into a class

### 5.1 `PoolDataset` — section 4 in `__getitem__`

`PoolDataset` stores the pools and settings; `__getitem__(i)` runs exactly the steps of section 4 for pool `i`. The
random generator decides the subset and query:
- **train**: seeded with `(seed, epoch, i)` → a new draw every epoch, but reproducible, and independent of which
  DataLoader worker builds the graph;
- **dev** (`fixed_draws=True`): seeded with `i` → pool `i` always gives the same graph, so the dev loss of different
  epochs is comparable.

In [90]:
class PoolDataset(torch.utils.data.Dataset):
    def __init__(self, pools, stats, k_graph, subset_min, subset_max, min_graph_size, use_gse, fixed_draws=False, seed=0):
        self.seed = seed
        self.epoch = 0
        self.pools = pools
        self.stats = stats
        self.k_graph = k_graph
        self.subset_min = subset_min
        self.subset_max = subset_max
        self.min_graph_size = min_graph_size
        self.use_gse = use_gse
        self.fixed_draws = fixed_draws

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.pools)

    def __getitem__(self, i):
        pool = self.pools[i]
        n = len(pool["vel"])
        lat, lon = pool["lat"], pool["lon"]
        rng = np.random.default_rng(i) if self.fixed_draws else np.random.default_rng([self.seed, self.epoch, i])
        idx, q = draw_subset(n, self.subset_min, self.subset_max, self.min_graph_size, rng)
        dist, az = pairwise_geodesic(lat[idx], lon[idx])
        edge_index = knn_edges(dist, self.k_graph)
        x = build_node_features(pool, idx, q, self.stats, self.use_gse)
        edge_attr = build_edge_features(dist, az, edge_index, self.stats)
        y = torch.tensor([(pool["vel"][idx][q] - self.stats["mean_velocity"]["mean"]) / self.stats["mean_velocity"]["std"]], dtype=torch.float32)
        query_mask = torch.zeros(len(idx), dtype=torch.bool)
        query_mask[q] = True
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, query_mask=query_mask,
            region=pool["region"], orbit=pool["orbit"])

In [91]:
# __init__ only stores the arguments as attributes (self.pools = pools, ...). Nothing is computed: the graphs are only
# built when an item is requested
train_ds = PoolDataset(train_pools, stats, K, 0.6, 0.9, MIN_GRAPH_SIZE, USE_GSE, seed=42)
print("attributes:", list(vars(train_ds))[:6], "...")
print("train_ds.epoch =", train_ds.epoch, "| train_ds.fixed_draws =", train_ds.fixed_draws)

attributes: ['seed', 'epoch', 'pools', 'stats', 'k_graph', 'subset_min'] ...
train_ds.epoch = 0 | train_ds.fixed_draws = False


In [92]:
# __len__: the number of pools. The DataLoader uses it to know how many items one epoch has.
# len(train_ds) calls train_ds.__len__() behind the scenes
print("len(train_ds) =", len(train_ds), "-> one graph per pool and epoch")

len(train_ds) = 14 -> one graph per pool and epoch


In [93]:
# __getitem__, the random generator line:
# rng = np.random.default_rng(i) if self.fixed_draws else np.random.default_rng([self.seed, self.epoch, i])
# a LIST as seed: numpy mixes all three numbers into one seed. Same triple -> same numbers, any change -> different
for seed_, epoch_, i_ in [(42, 0, 0), (42, 0, 0), (42, 1, 0), (42, 0, 1), (7, 0, 0)]:
    print(f"default_rng([{seed_}, {epoch_}, {i_}]).random() = {np.random.default_rng([seed_, epoch_, i_]).random():.6f}")

default_rng([42, 0, 0]).random() = 0.773956
default_rng([42, 0, 0]).random() = 0.773956
default_rng([42, 1, 0]).random() = 0.793503
default_rng([42, 0, 1]).random() = 0.866080
default_rng([7, 0, 0]).random() = 0.625095


In [95]:
# the remaining lines of __getitem__ are section 4: draw_subset -> pairwise_geodesic -> knn_edges -> node features
# -> edge features -> y and query_mask -> Data. train_ds[0] calls train_ds.__getitem__(0) behind the scenes
a, b = train_ds[0], train_ds[0]
print("same epoch, same pool twice -> identical:", torch.equal(a.x, b.x) and torch.equal(a.edge_index, b.edge_index))

same epoch, same pool twice -> identical: True


In [96]:
# set_epoch(epoch): train_model calls it before every epoch, so the next draws belong to the new epoch
train_ds.set_epoch(1)
c = train_ds[0]
print("epoch 1:", c.num_nodes, "nodes vs epoch 0:", a.num_nodes, "| identical:", c.num_nodes == a.num_nodes and torch.equal(c.x, a.x))
train_ds.set_epoch(0)

epoch 1: 16 nodes vs epoch 0: 16 | identical: False


In [98]:
# dev: fixed_draws=True -> the same graph in every epoch
dev_pools = load_pools(build_pool_index(dev_files), USE_GSE, MIN_GRAPH_SIZE)
dev_ds = PoolDataset(dev_pools, stats, K, 0.6, 0.9, MIN_GRAPH_SIZE, USE_GSE, fixed_draws=True)
a = dev_ds[0]; dev_ds.set_epoch(5); b = dev_ds[0]
print("dev pools:", len(dev_ds), "| epoch 0 vs epoch 5 identical:", torch.equal(a.x, b.x) and torch.equal(a.edge_index, b.edge_index))

dev pools: 6 | epoch 0 vs epoch 5 identical: True


### 5.2 `estimate_edge_stats` — mean and std of the log edge distances

The edge distances depend on the draws, so their stats are estimated from many random train graphs. Must run before the
dataset has `edge_dist` in its stats, so the drawn graphs still carry the raw `log(1 + d)`.

In [99]:
def estimate_edge_stats(dataset, n_draws=20000):
    graphs = [dataset[i] for i in np.random.default_rng(0).integers(0, len(dataset), n_draws)]
    d = torch.cat([g.edge_attr[:, 0] for g in graphs]).numpy()
    stats_d = {"mean": float(d.mean()), "std": float(d.std())}
    return {**dataset.stats, "edge_dist": stats_d}

In [100]:
# np.random.default_rng(0).integers(0, len(dataset), n_draws): n_draws random pool numbers 0 ... len-1.
# integers draws WITH replacement, so a pool can come up several times (s04 uses 20000 draws, here 2000)
dataset, n_draws = train_ds, 2000
positions = np.random.default_rng(0).integers(0, len(dataset), n_draws)
show("positions (first 20)", positions[:20])

positions (first 20): ndarray | dtype int64 | shape (20,)
[11  8  7  3  4  0  1  0  2 11  9 12  7  8 13 10  8  7  7 13] 



In [101]:
# graphs = [dataset[i] for i in positions]: builds one graph per drawn number (calls __getitem__)
graphs = [dataset[i] for i in positions]
print(len(graphs), "graphs, the first one:", graphs[0])

2000 graphs, the first one: Data(x=[9, 67], edge_index=[2, 36], edge_attr=[36, 3], y=[1], query_mask=[9], region='RegionA', orbit='descending')


In [102]:
# g.edge_attr[:, 0]: column 0 = the raw log(1 + d) of all edges of one graph
# torch.cat([...]): all edges of all graphs into one long vector; .numpy(): to numpy for mean/std
print("graph 0, log distances:", graphs[0].edge_attr[:5, 0])
d = torch.cat([g.edge_attr[:, 0] for g in graphs]).numpy()
show("d (all edges of all drawn graphs)", d[:10])
print("number of edges in total:", len(d))

graph 0, log distances: tensor([6.5300, 6.2442, 4.6637, 6.4522, 6.3547])
d (all edges of all drawn graphs): ndarray | dtype float32 | shape (10,)
[6.53   6.2442 4.6637 6.4522 6.3547 6.53   6.7464 6.4348 6.1539 6.8207] 

number of edges in total: 90116


In [103]:
# stats_d: mean and std of the log distance. float(...) makes plain python numbers (json-serialisable for the checkpoint)
stats_d = {"mean": float(d.mean()), "std": float(d.std())}
print(stats_d, "| in metres, exp(mean) - 1 =", round(np.expm1(stats_d["mean"]), 1), "m")

{'mean': 6.106091022491455, 'std': 0.6807672381401062} | in metres, exp(mean) - 1 = 447.6 m


In [104]:
# {**dataset.stats, "edge_dist": stats_d}: ** unpacks the old dict into a new one, then the new key is added.
# The old stats dict is not changed
new_stats = {**dataset.stats, "edge_dist": stats_d}
print("old keys:", list(dataset.stats), "\nnew keys:", list(new_stats))

old keys: ['mean_velocity', 'incidence_angle'] 
new keys: ['mean_velocity', 'incidence_angle', 'edge_dist']


In [105]:
# in build_datasets the new stats are given to both datasets; from now on the edge distance is normalized as well
stats = new_stats
train_ds.stats = dev_ds.stats = stats
print("edge distance column now:", train_ds[0].edge_attr[:5, 0].numpy(), "(mean ~0, std ~1 over many draws)")

edge distance column now: [-0.9787 -1.6325 -0.6584 -1.3108 -2.0085] (mean ~0, std ~1 over many draws)


## 6. Batching — what the `DataLoader` makes of several graphs

PyG combines a batch of graphs into **one big disconnected graph**: the node tables are stacked, the edge indices are
shifted so they point to the right rows, and `batch.batch` says which graph every node belongs to. Strings like `region`
become lists. Every graph has exactly one query, so `query_mask` selects exactly one node per graph.

In [106]:
# DataLoader(dataset, batch_size, shuffle): calls __getitem__ batch_size times and glues the graphs together.
# shuffle=True: a new random order of the pools every epoch (train); dev uses shuffle=False
loader = DataLoader(train_ds, batch_size=4, shuffle=True)
batch = next(iter(loader))      # iter() starts one pass over the loader, next() takes its first batch
print(batch)

DataBatch(x=[46, 67], edge_index=[2, 194], edge_attr=[194, 3], y=[4], query_mask=[46], region=[4], orbit=[4], batch=[46], ptr=[5])


In [107]:
print("num_graphs:", batch.num_graphs)
# batch.batch: the graph id of every node; bincount counts how many nodes each graph has
print("nodes per graph:", torch.bincount(batch.batch).tolist(), "| total nodes:", batch.num_nodes)
print("batch.batch:", batch.batch.tolist())
# batch.ptr: where every graph starts in the stacked node table (and the end as last value)
print("ptr (first node of every graph):", batch.ptr.tolist())

num_graphs: 4
nodes per graph: [16, 7, 9, 14] | total nodes: 46
batch.batch: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
ptr (first node of every graph): [0, 16, 23, 32, 46]


In [108]:
print("x:", tuple(batch.x.shape), "= all nodes of all graphs stacked")
print("edge_index:", tuple(batch.edge_index.shape), "| edge_attr:", tuple(batch.edge_attr.shape))
print("y:", tuple(batch.y.shape), "-> one target per graph | queries selected by query_mask:", int(batch.query_mask.sum()))
print("region / orbit (lists):", batch.region, batch.orbit)

x: (46, 67) = all nodes of all graphs stacked
edge_index: (2, 194) | edge_attr: (194, 3)
y: (4,) -> one target per graph | queries selected by query_mask: 4
region / orbit (lists): ['RegionB', 'RegionA', 'RegionA', 'RegionB'] ['ascending', 'ascending', 'descending', 'ascending']


In [110]:
# the edge indices of graph 1 are shifted by the number of nodes of graph 0 (= ptr[1]), so they point to its rows
g1_edges = batch.edge_index[:, batch.batch[batch.edge_index[0]] == 1]
print("edges of graph 1 point to rows", g1_edges.min().item(), "...", g1_edges.max().item(),
      "| its nodes are rows", batch.ptr[1].item(), "...", batch.ptr[2].item() - 1)

edges of graph 1 point to rows 16 ... 22 | its nodes are rows 16 ... 22


## 7. The model — `InSARGNN`

Follow the shapes: `(nodes, features)` → `(nodes, hidden)` through the encoder and the message passing layers →
`(graphs, hidden)` at the query nodes → one `mu` and one `logvar` per graph.

In [111]:
class InSARGNN(nn.Module):
    def __init__(self, n_node_features, edge_dim, hidden_dim, n_layers, heads, dropout):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_node_features, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        self.convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=edge_dim, dropout=dropout)
            for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.head_mu = nn.Linear(hidden_dim, 1)
        self.head_logvar = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        h = self.encoder(data.x)
        for conv, norm in zip(self.convs, self.norms):
            h = norm(h + nn.functional.gelu(conv(h, data.edge_index, data.edge_attr)))
        h_query = h[data.query_mask]
        mu = self.head_mu(h_query).squeeze(-1)
        logvar = self.head_logvar(h_query).squeeze(-1).clamp(-8, 8)
        return mu, logvar

**`__init__` line by line.** `super().__init__()` sets up `nn.Module`; every layer assigned to `self` afterwards is registered, so its weights are trained, moved to the GPU and saved.

In [112]:
n_node_features, edge_dim = train_ds[0].x.shape[1], train_ds[0].edge_attr.shape[1]
hidden_dim, n_layers, heads, dropout = 32, 2, 4, 0.1
torch.manual_seed(0)
model = InSARGNN(n_node_features, edge_dim, hidden_dim, n_layers, heads, dropout)
print("n_node_features =", n_node_features, "| edge_dim =", edge_dim)

n_node_features = 67 | edge_dim = 3


In [113]:
# self.encoder = nn.Sequential(Linear, GELU, Linear): a small MLP that lifts the 67 (or 3) raw features to hidden_dim.
# nn.Linear(a, b) computes x @ W.T + b with a weight matrix W of shape (b, a)
print(model.encoder)
print("weight of the first Linear:", tuple(model.encoder[0].weight.shape), "= (hidden_dim, n_node_features)")

Sequential(
  (0): Linear(in_features=67, out_features=32, bias=True)
  (1): GELU(approximate='none')
  (2): Linear(in_features=32, out_features=32, bias=True)
)
weight of the first Linear: (32, 67) = (hidden_dim, n_node_features)


In [114]:
# self.convs = nn.ModuleList([GATv2Conv(hidden_dim, hidden_dim // heads, heads=heads, ...) for _ in range(n_layers)])
# every head produces hidden_dim // heads values per node; the heads are concatenated -> hidden_dim again.
# nn.ModuleList instead of a python list: a plain list would hide the layers from nn.Module (not trained, not saved)
print(model.convs[0])
print("output per head:", hidden_dim // heads, "x", heads, "heads =", hidden_dim)

GATv2Conv(32, 8, heads=4)
output per head: 8 x 4 heads = 32


In [115]:
# self.norms: one LayerNorm per layer; self.head_mu / self.head_logvar: one number each per query node
print(model.norms[0], "|", model.head_mu, "|", model.head_logvar)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True) | Linear(in_features=32, out_features=1, bias=True) | Linear(in_features=32, out_features=1, bias=True)
trainable parameters: 7970


**`forward` line by line** on the batch of section 6. `model.eval()` switches the dropout off, so the step-by-step result can be compared with `model(batch)` exactly.

In [116]:
model.eval()
data = batch
# h = self.encoder(data.x): the same MLP is applied to every node (every row) independently
h = model.encoder(data.x)
print("x:", tuple(data.x.shape), "-> h:", tuple(h.shape))

x: (46, 67) -> h: (46, 32)


In [117]:
# conv(h, edge_index, edge_attr): message passing. Every node collects the h of its neighbours, weighted by attention
# scores that depend on both nodes and on the edge features (distance, bearing). GATv2Conv adds a self loop to every
# node (so a node also attends to itself). return_attention_weights=True also returns these weights
conv, norm = model.convs[0], model.norms[0]
out, (ei_att, alpha) = conv(h, data.edge_index, data.edge_attr, return_attention_weights=True)
print("conv output:", tuple(out.shape), "| attention:", tuple(alpha.shape), "= (edges incl. self loops, heads)")
qnode = int(data.query_mask.nonzero()[0])
into_q = ei_att[1] == qnode
print(f"edges into the query node {qnode} come from nodes", ei_att[0, into_q].tolist())
print("their attention weights per head (columns):\n", alpha[into_q].detach().numpy().round(3))
print("per head they sum to 1 (softmax over the incoming edges):", alpha[into_q].sum(0).detach().numpy().round(3))

conv output: (46, 32) | attention: (240, 4) = (edges incl. self loops, heads)
edges into the query node 8 come from nodes [1, 9, 15, 8]
their attention weights per head (columns):
 [[0.256 0.268 0.256 0.231]
 [0.267 0.263 0.269 0.23 ]
 [0.231 0.218 0.237 0.279]
 [0.246 0.251 0.238 0.26 ]]
per head they sum to 1 (softmax over the incoming edges): [1. 1. 1. 1.]


In [118]:
# nn.functional.gelu(...): a smooth non-linearity (like ReLU, but without the hard kink at 0)
act = nn.functional.gelu(out)
# h + act: the residual connection. The layer only adds a correction to h, so the node's own information is kept
res = h + act
# norm(...): LayerNorm normalizes every node's vector to mean ~0 and std ~1 (then a learned scale and shift)
h1 = norm(res)
print("row means after LayerNorm:", h1.mean(1)[:4].detach().numpy().round(3))
print("row stds after LayerNorm: ", h1.std(1, unbiased=False)[:4].detach().numpy().round(3))

row means after LayerNorm: [-0.  0.  0.  0.]
row stds after LayerNorm:  [1. 1. 1. 1.]


In [119]:
# the for loop does this for every layer (zip pairs every conv with its norm)
h = model.encoder(data.x)
for layer, (conv, norm) in enumerate(zip(model.convs, model.norms)):
    h = norm(h + nn.functional.gelu(conv(h, data.edge_index, data.edge_attr)))
    print(f"after layer {layer}:", tuple(h.shape))

after layer 0: (46, 32)
after layer 1: (46, 32)


In [120]:
# h_query = h[data.query_mask]: boolean indexing keeps only the rows where the mask is True -> one row per graph
h_query = h[data.query_mask]
print("h:", tuple(h.shape), "-> h_query:", tuple(h_query.shape))

h: (46, 32) -> h_query: (4, 32)


In [121]:
# self.head_mu(h_query) has shape (n_graphs, 1); .squeeze(-1) removes the last dimension of size 1 -> (n_graphs,)
raw_mu = model.head_mu(h_query)
mu = raw_mu.squeeze(-1)
print("before squeeze:", tuple(raw_mu.shape), "| after:", tuple(mu.shape))
# .clamp(-8, 8): keeps logvar in [-8, 8] so exp(logvar) can never become 0 or infinite
logvar = model.head_logvar(h_query).squeeze(-1).clamp(-8, 8)
print("mu:", mu.detach().numpy(), "\nlogvar:", logvar.detach().numpy())
mu_f, logvar_f = model(batch)
print("check, same as model(batch):", torch.allclose(mu, mu_f) and torch.allclose(logvar, logvar_f))

before squeeze: (4, 1) | after: (4,)
mu: [-0.4289 -0.5292 -0.3885 -0.4383] 
logvar: [-1.2633 -1.2612 -1.1841 -1.3117]
check, same as model(batch): True


## 8. The loss — `loss_fn`

Heteroscedastic Gaussian NLL: `0.5 · (logvar + (y − mu)² / exp(logvar))`. A large error is cheaper if the model also says
"I'm unsure" (large `logvar`), but being unsure costs `logvar` itself.

In [122]:
def loss_fn(mu, logvar, y):
    loss = 0.5 * (logvar + (y - mu) ** 2 / torch.exp(logvar))
    return loss.mean()

In [123]:
# three toy queries: right and confident, wrong and confident, wrong but unsure
y_ = torch.tensor([1.0, 1.0, 1.0])
mu_ = torch.tensor([1.0, 0.0, 0.0])
logvar_ = torch.tensor([-3.0, -3.0, 1.0])
print("y - mu            :", (y_ - mu_))
print("(y - mu) ** 2     :", (y_ - mu_) ** 2, " <- squared error")
print("exp(logvar)       :", torch.exp(logvar_), " <- the predicted variance sigma^2")
print("error / variance  :", (y_ - mu_) ** 2 / torch.exp(logvar_), " <- the error measured in units of the predicted variance")
per_query = 0.5 * (logvar_ + (y_ - mu_) ** 2 / torch.exp(logvar_))
print("0.5 * (logvar + .):", per_query, " <- logvar is the price for being unsure")
print(".mean()           :", per_query.mean(), "| check:", torch.isclose(per_query.mean(), loss_fn(mu_, logvar_, y_)).item())

y - mu            : tensor([0., 1., 1.])
(y - mu) ** 2     : tensor([0., 1., 1.])  <- squared error
exp(logvar)       : tensor([0.0498, 0.0498, 2.7183])  <- the predicted variance sigma^2
error / variance  : tensor([ 0.0000, 20.0855,  0.3679])  <- the error measured in units of the predicted variance
0.5 * (logvar + .): tensor([-1.5000,  8.5428,  0.6839])  <- logvar is the price for being unsure
.mean()           : tensor(2.5756) | check: True


In [124]:
y1 = torch.tensor([1.0])
for m_, lv in [(1.0, 0.0), (1.0, -3.0), (0.0, 0.0), (0.0, -3.0), (0.0, 1.0), (0.0, 3.0)]:
    print(f"error {abs(1 - m_):.0f}, logvar {lv:4.1f} (sigma {np.exp(lv / 2):5.2f}) -> loss {loss_fn(torch.tensor([m_]), torch.tensor([lv]), y1).item():6.3f}")
print("-> right and confident is best; wrong and confident is worst; wrong but unsure is in between")

error 0, logvar  0.0 (sigma  1.00) -> loss  0.000
error 0, logvar -3.0 (sigma  0.22) -> loss -1.500
error 1, logvar  0.0 (sigma  1.00) -> loss  0.500
error 1, logvar -3.0 (sigma  0.22) -> loss  8.543
error 1, logvar  1.0 (sigma  1.65) -> loss  0.684
error 1, logvar  3.0 (sigma  4.48) -> loss  1.525
-> right and confident is best; wrong and confident is worst; wrong but unsure is in between


## 9. Training

### 9.1 `set_seed` — one seed for python, numpy and torch

In [125]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

In [126]:
# every library has its own random generator; set_seed seeds all of them. Same seed -> same "random" numbers,
# e.g. the same initial weights of the model
set_seed(42); a1 = (random.random(), np.random.rand(), torch.rand(1).item())
set_seed(42); a2 = (random.random(), np.random.rand(), torch.rand(1).item())
print("python, numpy, torch:", a1, "\nagain with seed 42: ", a2, "| identical:", a1 == a2)

python, numpy, torch: (0.6394267984578837, 0.3745401188473625, 0.8822692632675171) 
again with seed 42:  (0.6394267984578837, 0.3745401188473625, 0.8822692632675171) | identical: True


### 9.2 `train_one_epoch` — one pass over the train loader

In [127]:
def train_one_epoch(model, loader, optimizer, grad_clip, device):
    model.train()
    total_loss, n_graphs = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        mu, logvar = model(batch)
        loss = loss_fn(mu, logvar, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        n_graphs += batch.num_graphs
    return total_loss / n_graphs

**Line by line** for one batch.

In [128]:
device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
model = InSARGNN(n_node_features, edge_dim, hidden_dim=32, n_layers=2, heads=4, dropout=0.1).to(device)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=4, shuffle=False)
# AdamW: the optimizer, it gets the parameters it is allowed to change, the learning rate and the weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
grad_clip = 5.0

In [129]:
# model.train(): training mode -> dropout is active (model.eval() switches it off)
model.train()
print("model.training =", model.training)
total_loss, n_graphs = 0.0, 0

model.training = True


In [130]:
# for batch in loader: batch = batch.to(device) -> all tensors of the batch to the GPU (here: wherever device is)
batch = next(iter(train_loader)).to(device)
print(batch)

DataBatch(x=[44, 67], edge_index=[2, 178], edge_attr=[178, 3], y=[4], query_mask=[44], region=[4], orbit=[4], batch=[44], ptr=[5])


In [131]:
# optimizer.zero_grad(): deletes the gradients of the previous batch. PyTorch ADDS new gradients to existing ones,
# so without this line every step would also contain all earlier batches
optimizer.zero_grad()
w = model.head_mu.weight
print("gradient of head_mu.weight before backward:", w.grad)

gradient of head_mu.weight before backward: None


In [132]:
# forward pass and loss: what the model predicts for the queries and how wrong (and how honest) that is
mu, logvar = model(batch)
loss = loss_fn(mu, logvar, batch.y)
print("mu:", mu.detach().numpy(), "\ny: ", batch.y.numpy(), "\nloss:", loss.item())

mu: [0.1467 0.5811 0.6309 0.4838] 
y:  [-1.1651 -1.3249 -0.1698 -0.151 ] 
loss: 0.7737877368927002


In [133]:
# loss.backward(): backpropagation, computes d(loss)/d(weight) for every weight and stores it in weight.grad
loss.backward()
print("gradient of head_mu.weight:", tuple(w.grad.shape), "| norm:", w.grad.norm().item())

gradient of head_mu.weight: (1, 32) | norm: 5.227532386779785


In [134]:
# clip_grad_norm_(parameters, max_norm): if the length of ALL gradients together (one long vector) is above
# grad_clip, they are scaled down to exactly that length. Protects against single huge steps. Returns the length before
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
print("total gradient norm before clipping:", total_norm.item(), "| clipped:", total_norm.item() > grad_clip)

total gradient norm before clipping: 9.431074142456055 | clipped: True


In [135]:
# optimizer.step(): changes every weight a little against its gradient (AdamW's update rule)
before = w.detach().clone()
optimizer.step()
print("largest change of a head_mu weight:", (w.detach() - before).abs().max().item())

largest change of a head_mu weight: 0.0030000507831573486


In [136]:
# loss.item(): the tensor as a plain python float. loss is the MEAN over the batch, so it is multiplied by the batch
# size before summing; at the end total / n_graphs = mean per graph, and a smaller last batch counts less
total_loss += loss.item() * batch.num_graphs
n_graphs += batch.num_graphs
print("total_loss =", total_loss, "| n_graphs =", n_graphs, "| mean so far =", total_loss / n_graphs)

total_loss = 3.095150947570801 | n_graphs = 4 | mean so far = 0.7737877368927002


In [137]:
# the whole function: all batches of one epoch
print("mean train loss of one epoch:", train_one_epoch(model, train_loader, optimizer, grad_clip, device))

mean train loss of one epoch: 0.7657495481627328


### 9.3 `evaluate` — predictions and metrics without training

In [138]:
def evaluate(model, loader, device):
    model.eval()
    with torch.no_grad():
        mus, logvars, ys, regions, orbits = [], [], [], [], []
        for batch in loader:
            batch = batch.to(device)
            mu, logvar = model(batch)
            mus.append(mu.cpu()); logvars.append(logvar.cpu()); ys.append(batch.y.cpu())
            regions.extend(batch.region); orbits.extend(batch.orbit)
        mu, logvar, y = torch.cat(mus), torch.cat(logvars), torch.cat(ys)
        nll = loss_fn(mu, logvar, y).item()
        v = loader.dataset.stats["mean_velocity"]
        mu_mm = mu * v["std"] + v["mean"]
        y_mm = y * v["std"] + v["mean"]
        sigma_mm = torch.exp(0.5 * logvar) * v["std"]
        err = mu_mm - y_mm
        rmse = torch.sqrt((err ** 2).mean()).item()
        mae = err.abs().mean().item()
        inside = (err.abs() <= 1.645 * sigma_mm).float().mean().item()
        metrics = {"nll": nll, "rmse": rmse, "mae": mae, "inside_90": inside}
        predictions = pl.DataFrame({"region": regions, "orbit": orbits, "y_mm": y_mm.numpy(),
                            "mu_mm": mu_mm.numpy(), "sigma_mm": sigma_mm.numpy()})
    return metrics, predictions

In [139]:
# model.eval(): dropout off. torch.no_grad(): no gradients are recorded (faster, less memory, nothing to train here)
model.eval()
with torch.no_grad():
    out_mu, _ = model(next(iter(dev_loader)).to(device))
print("model.training =", model.training, "| output records gradients:", out_mu.requires_grad)

model.training = False | output records gradients: False


In [140]:
# the loop collects the outputs of every batch in lists. .cpu(): back from the GPU. extend (not append) adds the
# single strings of batch.region instead of the whole list as one element
mus, logvars, ys, regions, orbits = [], [], [], [], []
with torch.no_grad():
    for b in dev_loader:
        b = b.to(device)
        m_, lv_ = model(b)
        mus.append(m_.cpu()); logvars.append(lv_.cpu()); ys.append(b.y.cpu())
        regions.extend(b.region); orbits.extend(b.orbit)
print("batches:", len(mus), "| shapes per batch:", [tuple(t.shape) for t in mus], "| regions:", len(regions))

batches: 2 | shapes per batch: [(4,), (2,)] | regions: 6


In [141]:
# torch.cat: the per-batch tensors into one tensor -> one entry per query (not per batch)
mu, logvar, y = torch.cat(mus), torch.cat(logvars), torch.cat(ys)
print("mu:", tuple(mu.shape), "| nll (normalized units):", loss_fn(mu, logvar, y).item())

mu: (6,) | nll (normalized units): 0.6994479298591614


In [142]:
# back to mm/yr: undo the normalization. sigma = exp(0.5 * logvar) = sqrt(variance), scaled with the std as well
v = dev_loader.dataset.stats["mean_velocity"]
mu_mm = mu * v["std"] + v["mean"]
y_mm = y * v["std"] + v["mean"]
sigma_mm = torch.exp(0.5 * logvar) * v["std"]
print("mu_mm:", mu_mm.numpy()[:4], "\ny_mm: ", y_mm.numpy()[:4], "\nsigma:", sigma_mm.numpy()[:4])

mu_mm: [-11.2577  -9.6391  -9.5971 -11.6497] 
y_mm:  [ -4.0256 -15.6073  -4.2252  -3.8139] 
sigma: [5.0158 6.2845 6.1646 5.4753]


In [143]:
# the metrics: RMSE punishes large errors more (squared), MAE is the typical error.
# inside_90: share of queries whose true value lies within mu +- 1.645 sigma (the 90 % interval of a Gaussian)
err = mu_mm - y_mm
print("err:", err.numpy()[:4])
print("rmse:", torch.sqrt((err ** 2).mean()).item(), "| mae:", err.abs().mean().item())
inside_mask = err.abs() <= 1.645 * sigma_mm
print("inside the interval:", inside_mask.tolist()[:8], "-> .float().mean() =", inside_mask.float().mean().item())

err: [-7.2321  5.9682 -5.3719 -7.8359]
rmse: 6.238770961761475 | mae: 6.028602123260498
inside the interval: [True, True, True, True, True, True] -> .float().mean() = 1.0


In [144]:
# the predictions table: one row per query, used for the per-region and per-orbit metrics
metrics, dev_predictions = evaluate(model, dev_loader, device)
print({k: round(v_, 3) for k, v_ in metrics.items()})
dev_predictions.head()

{'nll': 0.699, 'rmse': 6.239, 'mae': 6.029, 'inside_90': 1.0}


region,orbit,y_mm,mu_mm,sigma_mm
str,str,f32,f32,f32
"""RegionC""","""ascending""",-4.025637,-11.257748,5.015831
"""RegionC""","""ascending""",-15.607325,-9.639139,6.284474
"""RegionC""","""ascending""",-4.22518,-9.597054,6.164625
"""RegionC""","""ascending""",-3.813871,-11.649735,5.475282
"""RegionC""","""ascending""",-4.242134,-11.084714,5.449601


### 9.4 `save_checkpoint` and `train_model` — the epoch loop with early stopping

Every epoch: new train draws (`set_epoch`), one training pass, one evaluation on dev. A new best dev NLL saves a
checkpoint; after `patience` epochs without a new best the training stops.

In [ ]:
def save_checkpoint(model, params, epoch, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "params": params, "epoch": epoch}, path)

def train_model(model, train_loader, dev_loader, lr, weight_decay, max_epochs, patience, grad_clip,
                device, checkpoint_path, params):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_loss, best_epoch, epochs_without_improvement, history = float("inf"), None, 0, []
    for epoch in range(max_epochs):
        train_loader.dataset.set_epoch(epoch)
        train_loss = train_one_epoch(model, train_loader, optimizer, grad_clip, device)
        dev_metrics, _ = evaluate(model, dev_loader, device)
        history.append({"epoch": epoch, "train_loss": train_loss,
                "dev_loss": dev_metrics["nll"], "dev_rmse": dev_metrics["rmse"]})
        print(history[-1], flush=True)
        if dev_metrics["nll"] < best_loss:
            best_loss, best_epoch = dev_metrics["nll"], epoch
            epochs_without_improvement = 0
            save_checkpoint(model, params, epoch, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    return history, best_epoch

In [ ]:
# save_checkpoint: model.state_dict() = all weights as an ordered dict {layer name: tensor}. Saved together with the
# params (to rebuild the model later) and the epoch. mkdir(parents=True, exist_ok=True): create missing folders, no
# error if they exist
sd = model.state_dict()
print("state_dict:", type(sd).__name__, "with", len(sd), "tensors, e.g.", list(sd)[:3])

In [ ]:
# the bookkeeping of the early stopping:
# best_loss = inf so that the first epoch is always an improvement; best_epoch = None until then
best_loss, best_epoch, epochs_without_improvement, history = float("inf"), None, 0, []
print("float('inf') > any number:", float("inf") > 1e30)

In [ ]:
# the whole training on the toy data (small patience so you can see it stop)
checkpoint_path = Path(tempfile.mkdtemp()) / "checkpoints" / "toy_run" / "best_model.pt"
set_seed(42)
model = InSARGNN(n_node_features, edge_dim, hidden_dim=32, n_layers=2, heads=4, dropout=0.1).to(device)
history, best_epoch = train_model(model, train_loader, dev_loader, lr=3e-3, weight_decay=1e-4, max_epochs=60,
                                  patience=8, grad_clip=5.0, device=device, checkpoint_path=checkpoint_path,
                                  params={"stats": stats})
print("epochs run:", len(history), "| best epoch:", best_epoch)

In [ ]:
import matplotlib.pyplot as plt
h_ = pl.DataFrame(history)
plt.plot(h_["epoch"], h_["train_loss"], label="train"); plt.plot(h_["epoch"], h_["dev_loss"], label="dev")
plt.axvline(best_epoch, ls="--", c="grey", label="best epoch"); plt.xlabel("epoch"); plt.ylabel("NLL"); plt.legend(); plt.show()

### 9.5 `evaluate_on_dev` — reload the best epoch and compute the metrics per group

In [ ]:
# torch.load: the saved dict back. map_location=device: load the tensors directly onto this device
checkpoint = torch.load(checkpoint_path, map_location=device)
print("checkpoint keys:", list(checkpoint), "| saved at epoch", checkpoint["epoch"])
# load_state_dict: overwrites the current (last-epoch) weights with the best ones
model.load_state_dict(checkpoint["state_dict"])
train_metrics, _ = evaluate(model, train_loader, device)
dev_metrics, dev_predictions = evaluate(model, dev_loader, device)
print("train:", {k: round(v_, 3) for k, v_ in train_metrics.items()})
print("dev:  ", {k: round(v_, 3) for k, v_ in dev_metrics.items()})

In [ ]:
# per_group(col): the error as a polars expression (computed per row once it is used), then per group:
# pl.len() = number of rows, (err ** 2).mean().sqrt() = RMSE, err.abs().mean() = MAE, the inside share as a mean of booleans
err = pl.col("mu_mm") - pl.col("y_mm")
per_region = (dev_predictions.group_by("region")
              .agg(pl.len().alias("n"),
                   (err ** 2).mean().sqrt().alias("rmse"),
                   err.abs().mean().alias("mae"),
                   (err.abs() <= 1.645 * pl.col("sigma_mm")).mean().alias("inside_90"))
              .sort("region"))
print(per_region)
# .to_dicts(): one dict per row, the format the markdown tables of the log are built from
print(per_region.to_dicts())

## Where to go from here

- Rerun from section 3 with `USE_GSE = False` and check that `x` has 3 instead of 67 columns.
- Change `K` (section 4.3) and watch the in-degree and the number of edges.
- Make the toy pools sparser (fewer points, larger spread in `make_graph_file`) and see how the edge distances and the
  dev metrics change.
- Change `hidden_dim`, `n_layers`, `heads` in section 7 and compare the parameter count.